<a href="https://colab.research.google.com/github/yhlimath/Spider-action-codes/blob/main/Checking_the_algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

README:

These codes check the braid-type (cubic) relations that the sl_3 quotient of Hecke algebra should obey. The Hecke (idempotent) relations are rather easy, and have been checked before, so they are not included here.

Specifically, the codes check the braid-type relations on all 462 state strings that generate an invariant tensor (thus we are living in the vacuum sector). On each state string the relation

e_i e_i+1 e_i - e_i = e_i+1 e_i e_i+1 - e_i+1

and its counterpart for f generator

are tested on several sites (3 for e and 8 for f, if I remember coorectly). All tests were correct so I'm convinced that the algorithm is correct.

TO RUN THE CODES, just press on "execute everything"/"tout exécuter" and wait for at most two minutes.

The AI wrote the codes and I checked them. Some tests they wrote were unnecessary but I kept them anyways.

# Preparatory codes: The functions and data structures

Technical part perparing necessary data structures

In [1]:
# the class FormalSumgives the space of strings a module structure over polynomials in n=[2]_q.

class FormalSum:
    """Class to represent formal linear combinations of strings with polynomial coefficients"""
    def __init__(self, terms=None):
        # terms is a dictionary: {tuple(string): Polynomial coefficient}
        self.terms = terms if terms is not None else {}
        self._normalize()

    def _normalize(self):
        """Remove terms with zero coefficients and combine like terms"""
        normalized_terms = {}
        for key, coeff in self.terms.items():
            if isinstance(coeff, (int, float)):
                if coeff != 0:
                    normalized_terms[key] = Polynomial.constant(coeff)
            elif isinstance(coeff, Polynomial):
                if not coeff.is_zero():
                    normalized_terms[key] = coeff
            else:
                # Handle other coefficient types
                normalized_terms[key] = coeff
        self.terms = normalized_terms

    def add_term(self, string, coeff=None):
        """Add a term to the formal sum"""
        if coeff is None:
            coeff = Polynomial.constant(1)
        elif isinstance(coeff, (int, float)):
            coeff = Polynomial.constant(coeff)

        key = tuple(string)
        if key in self.terms:
            self.terms[key] = self.terms[key] + coeff
        else:
            self.terms[key] = coeff
        self._normalize()

    def add_coefficient_string_pairs(self, pairs):
        """Add multiple (coefficient, string) pairs"""
        for coeff, string in pairs:
            self.add_term(string, coeff)

    def __add__(self, other):
        """Add two formal sums"""
        result = FormalSum()
        result.terms = {k: v for k, v in self.terms.items()}
        for key, coeff in other.terms.items():
            if key in result.terms:
                result.terms[key] = result.terms[key] + coeff
            else:
                result.terms[key] = coeff
        result._normalize()
        return result

    def __sub__(self, other):
        """Subtract two formal sums"""
        result = FormalSum()
        result.terms = {k: v for k, v in self.terms.items()}
        for key, coeff in other.terms.items():
            if key in result.terms:
                result.terms[key] = result.terms[key] - coeff
            else:
                result.terms[key] = -coeff
        result._normalize()
        return result

    def __eq__(self, other):
        """Check if two formal sums are equal"""
        return self.terms == other.terms

    def __str__(self):
        if not self.terms:
            return "0"
        parts = []
        for key, coeff in sorted(self.terms.items()):
            string = list(key)
            coeff_str = str(coeff)
            if coeff == Polynomial.constant(1):
                parts.append(f"{string}")
            elif coeff == Polynomial.constant(-1):
                parts.append(f"-{string}")
            else:
                parts.append(f"({coeff})*{string}")
        result = " + ".join(parts).replace("+ -", "- ")
        return result

    def __repr__(self):
        return f"FormalSum({self.terms})"

    def is_zero(self):
        """Check if the formal sum is zero"""
        return len(self.terms) == 0

def apply_e_to_formal_sum(formal_sum, i):
    """Apply operation e_i to a formal sum"""
    result = FormalSum()

    for string_tuple, coeff in formal_sum.terms.items():
        string = list(string_tuple)
        # Convert to the format expected by e function
        input_pairs = [(coeff, string)]
        # Apply e operation
        output_pairs = e(input_pairs, i, verbose=False)
        # Add results to formal sum
        result.add_coefficient_string_pairs(output_pairs)

    return result

def apply_sequence_to_formal_sum(formal_sum, operations):
    """Apply a sequence of e operations to a formal sum"""
    current = formal_sum
    for op in operations:
        current = apply_e_to_formal_sum(current, op)
    return current



def create_coefficient_string_pair(coeff, string):
    """Helper function to create (coefficient, string) pairs"""
    return (coeff, string)

def display_coefficient_strings(S):
    """Helper function to nicely display a list of (coefficient, string) pairs"""
    if not S:
        return "[]"
    parts = []
    for coeff, string in S:
        if coeff == 1:
            parts.append(f"{string}")
        elif coeff == -1:
            parts.append(f"-{string}")
        else:
            parts.append(f"{coeff}*{string}")
    return "[" + ", ".join(parts) + "]"


import random

class Polynomial:
    """Class to represent polynomials in variable n"""
    def __init__(self, coeffs=None):
        # coeffs is a dictionary: {power: coefficient}
        # e.g., n^2 + 3n - 1 would be {2: 1, 1: 3, 0: -1}
        self.coeffs = coeffs if coeffs is not None else {}
        self._normalize()

    def _normalize(self):
        """Remove terms with zero coefficients"""
        self.coeffs = {k: v for k, v in self.coeffs.items() if v != 0}

    @classmethod
    def constant(cls, value):
        """Create a constant polynomial"""
        return cls({0: value}) if value != 0 else cls({})

    @classmethod
    def variable(cls, power=1):
        """Create n^power"""
        return cls({power: 1})

    def __add__(self, other):
        """Add two polynomials"""
        if isinstance(other, (int, float)):
            other = Polynomial.constant(other)
        result = Polynomial()
        result.coeffs = self.coeffs.copy()
        for power, coeff in other.coeffs.items():
            result.coeffs[power] = result.coeffs.get(power, 0) + coeff
        result._normalize()
        return result

    def __radd__(self, other):
        return self.__add__(other)

    def __sub__(self, other):
        """Subtract two polynomials"""
        if isinstance(other, (int, float)):
            other = Polynomial.constant(other)
        result = Polynomial()
        result.coeffs = self.coeffs.copy()
        for power, coeff in other.coeffs.items():
            result.coeffs[power] = result.coeffs.get(power, 0) - coeff
        result._normalize()
        return result

    def __rsub__(self, other):
        return (Polynomial.constant(other) - self) if isinstance(other, (int, float)) else NotImplemented

    def __mul__(self, other):
        """Multiply two polynomials"""
        if isinstance(other, (int, float)):
            other = Polynomial.constant(other)
        result = Polynomial()
        for p1, c1 in self.coeffs.items():
            for p2, c2 in other.coeffs.items():
                power = p1 + p2
                result.coeffs[power] = result.coeffs.get(power, 0) + c1 * c2
        result._normalize()
        return result

    def __rmul__(self, other):
        return self.__mul__(other)

    def __neg__(self):
        """Negate a polynomial"""
        result = Polynomial()
        result.coeffs = {k: -v for k, v in self.coeffs.items()}
        return result

    def __eq__(self, other):
        """Check if two polynomials are equal"""
        if isinstance(other, (int, float)):
            other = Polynomial.constant(other)
        return self.coeffs == other.coeffs

    def __str__(self):
        if not self.coeffs:
            return "0"

        parts = []
        for power in sorted(self.coeffs.keys(), reverse=True):
            coeff = self.coeffs[power]

            if power == 0:
                if coeff > 0 and parts:
                    parts.append(f"+{coeff}")
                else:
                    parts.append(str(coeff))
            elif power == 1:
                if coeff == 1:
                    parts.append("+n" if parts else "n")
                elif coeff == -1:
                    parts.append("-n")
                else:
                    if coeff > 0 and parts:
                        parts.append(f"+{coeff}n")
                    else:
                        parts.append(f"{coeff}n")
            else:
                if coeff == 1:
                    parts.append(f"+n^{power}" if parts else f"n^{power}")
                elif coeff == -1:
                    parts.append(f"-n^{power}")
                else:
                    if coeff > 0 and parts:
                        parts.append(f"+{coeff}n^{power}")
                    else:
                        parts.append(f"{coeff}n^{power}")

        return "".join(parts)

    def __repr__(self):
        return f"Polynomial({self.coeffs})"

    def is_zero(self):
        """Check if polynomial is zero"""
        return len(self.coeffs) == 0

    def evaluate(self, n_value):
        """Evaluate polynomial at a specific value of n"""
        result = 0
        for power, coeff in self.coeffs.items():
            result += coeff * (n_value ** power)
        return result


MAIN CODES defining the generators e and f

In [2]:

def generate_balanced_string(n):
    """Generate a shuffled string of length 3n with equal numbers of 1, 0, and -1."""
    numbers = [1] * n + [0] * n + [-1] * n
    random.shuffle(numbers)
    return numbers

def generate_all_valid_strings(n):
    """Generate all valid strings of length 3n that correspond to walks in the sl_3 Weyl chamber """
    def backtrack(sequence, count_1, count_0, count_neg1, results):
        if len(sequence) == 3 * n:
            results.append(sequence[:])  # Store a copy of the valid sequence
            return
        if count_1 < n:
            backtrack(sequence + [1], count_1 + 1, count_0, count_neg1, results)
        if count_0 < count_1 and count_0 < n:
            backtrack(sequence + [0], count_1, count_0 + 1, count_neg1, results)
        if count_neg1 < count_0 and count_neg1 < n:
            backtrack(sequence + [-1], count_1, count_0, count_neg1 + 1, results)

    results = []
    backtrack([], 0, 0, 0, results)
    return results

def form_triplets_with_positions(sequence):
    """Form triplets from the sequence using the specified rightmost-1-first rule,
    and return the positions of the numbers in the original sequence."""
    sequence = sequence[:]
    indexed_sequence = list(enumerate(sequence))  # Store original positions
    triplets = []

    while indexed_sequence:
        try:
            index_1 = max(i for i, val in indexed_sequence if val == 1)
        except ValueError:
            break  # No more 1s left
        try:
            index_0 = next(i for i, val in indexed_sequence if val == 0 and i > index_1)
        except StopIteration:
            break  # No more 0s left
        try:
            index_neg1 = next(i for i, val in indexed_sequence if val == -1 and i > index_0)
        except StopIteration:
            break  # No more -1s left

        triplet = ((1, index_1), (0, index_0), (-1, index_neg1))
        triplets.append(triplet)

        indexed_sequence = [(i, val) for i, val in indexed_sequence if i not in {index_1, index_0, index_neg1}]

    return triplets

def bend_string(sequence):
    """Perform the bending operation on the given sequence."""
    sequence = sequence[:]
    indexed_sequence = list(enumerate(sequence))  # Store original positions

    try:
        start_1_index = next(i for i, val in indexed_sequence if val == 1)
    except StopIteration:
        return sequence  # No 1 found, return unchanged

    triplets = form_triplets_with_positions(sequence)
    for triplet in triplets:
        if triplet[0][0] == 1 and triplet[0][1] == start_1_index:
            index_1, index_0, index_neg1 = triplet[0][1], triplet[1][1], triplet[2][1]
            break
    else:
        return sequence  # No valid triplet found, return unchanged

    new_sequence = sequence[:]
    new_sequence[index_0] = 1
    new_sequence[index_neg1] = 0
    new_sequence.pop(index_1)
    new_sequence.append(-1)

    return new_sequence

def bending_power(s, p):
    """Applies the bending operation to string s for p times."""
    s_bent_p_times = s[:]  # Create a copy of s
    for _ in range(p):
        s_bent_p_times = bend_string(s_bent_p_times)
    return s_bent_p_times

def inverse_bending_power(s, p):
    """Applies the inverse bending operation to string s for p times."""
    return bending_power(s, len(s) - p)

def generate_triplet_at_position(s, p):
    if p > len(s):
        # If p is larger than the length of s, append the triplet at the end
        return s + [1, 0, -1]
    else:
        # Otherwise, insert the triplet at position p
        return s[:p-1] + [1, 0, -1] + s[p-1:]

def find_last_triplet(s):
    """Finds the last triplet (1, 0, -1) in the string s, containing the last 1."""
    try:
        last_1_index = len(s) - 1 - s[::-1].index(1)  # Find index of last 1
        last_0_index = next(i for i in range(last_1_index + 1, len(s)) if s[i] == 0)
        last_neg1_index = next(i for i in range(last_0_index + 1, len(s)) if s[i] == -1)
        return last_1_index, last_0_index, last_neg1_index
    except (ValueError, StopIteration):
        return None

def string_decomposition(s):
    s_decomposed = s[:]  # Create a copy of s to work with
    operations = []  # Initialize an empty list to store operations

    while len(s_decomposed) > 6:
        last_triplet_indices = find_last_triplet(s_decomposed)
        if last_triplet_indices is None:
            break  # No more triplets found, terminate

        last_1_index, last_0_index, last_neg1_index = last_triplet_indices

        if last_0_index == last_1_index + 1 and last_neg1_index == last_0_index + 1:
            # If last triplet occupies consecutive sites, delete it
            s_decomposed = s_decomposed[:last_1_index] + s_decomposed[last_neg1_index + 1:]
            operations.append(('t', last_1_index + 1))  # Record operation

        else:
            # Handle the case where the last triplet is not consecutive
            if all(x == -1 for x in s_decomposed[last_1_index + 1:last_0_index]):
                # If there is a string of -1s between 1 and 0
                start_neg1_string = last_1_index + 1
                end_neg1_string = last_0_index - 1

                # Perform cyclic permutation: 1, -1, ... , -1 to -1, ... , -1, 1
                s_decomposed[last_1_index], s_decomposed[end_neg1_string] = s_decomposed[end_neg1_string], s_decomposed[last_1_index]

                # Update operations with (e, position of -1 - 1) in increasing order of -1 positions
                for i in range(start_neg1_string, end_neg1_string + 1):
                    operations.append(('e', i))

                #Handle 0s between 0 and -1 simultaneously:
            if all(x == 0 for x in s_decomposed[last_0_index + 1:last_neg1_index]):
                    start_0_string = last_0_index + 1
                    end_0_string = last_neg1_index - 1

                    # Perform cyclic permutation for 0s:
                    s_decomposed[last_neg1_index], s_decomposed[start_0_string] = s_decomposed[start_0_string] ,s_decomposed[last_neg1_index]

                    # Record operations for 0s in decreasing order of positions:
                    for i in range(end_0_string, start_0_string -1, -1): #start_0_string included
                        operations.append(('e', i+1))

            else:
                # Handle other non-consecutive cases
                pass  # Placeholder for now

    return s_decomposed, operations

def e(S, i, verbose=False):
    """
    Apply operation e_i to a list of (coefficient, string) pairs.

    Args:
        S: List of (coefficient, string) pairs
        i: Position for the e operation
        verbose: Whether to print detailed output

    Returns:
        List of (coefficient, string) pairs
    """
    S_out = []

    for coeff, s in S:
        if verbose:
            print(f"Computing e({coeff} * {s}, {i})")

        bent_s = bending_power(s, i-1)
        if verbose:
            print(f"Bent string B^{i-1}({s}): {bent_s}")

        s_decomposed, operations = string_decomposition(bent_s)
        if verbose:
            print(f"Decomposing B^{i-1}({s}) gives s_decomposed= {s_decomposed} and operations={operations}")

        if s_decomposed in [[1, 0, 1, 0, -1, -1], [1, 0, 1, -1, 0, -1], [1, 0, -1, 1, 0, -1]]:
            # Return original string with coefficient multiplied by n
            if isinstance(coeff, (int, float)):
                new_coeff = Polynomial.constant(coeff) * Polynomial.variable()
            elif isinstance(coeff, Polynomial):
                new_coeff = coeff * Polynomial.variable()
            else:
                # Fallback for other types
                new_coeff = coeff

            S_out.append((new_coeff, s))
            if verbose:
                print(f"e({coeff} * {s}, {i}) -> {new_coeff} * {s} (ends at original string, coefficient multiplied by n)")

        elif s_decomposed == [1, 1, 0, -1, 0, -1]:
            # Find the triplet containing the 1 on the second entry
            triplets = form_triplets_with_positions(bent_s)
            target_triplet = None
            for triplet in triplets:
                if triplet[0][0] == 1 and triplet[0][1] == bent_s.index(1, 1):  # Find the second 1
                    target_triplet = triplet
                    break

            # Create s_jescon by swapping 1 and 0 in the target triplet
            if target_triplet:
                s_jescon = bent_s[:]
                index_1, index_0, _ = target_triplet[0][1], target_triplet[1][1], target_triplet[2][1]
                s_jescon[index_1], s_jescon[index_0] = s_jescon[index_0], s_jescon[index_1]
                result = inverse_bending_power(s_jescon, i - 1)
                S_out.append((coeff, result))
                if verbose:
                    print(f"e({coeff} * {s}, {i}) -> {coeff} * {result} (Jesper's conjecture)")

        elif s_decomposed == [1,1,0,0,-1,-1]:
            new_strings = [[1,0,-1,1,0,-1],[1,0,1,0,-1,-1]]
            if verbose:
                print(f"Action of e_1 creates a square here, s_decomposed splits into new strings")

            # Apply operations in reverse order to both new strings
            for op_type, position in reversed(operations):
                if op_type == 't':
                    temp_new_strings = []
                    for new_s in new_strings:
                        temp_new_strings.append(generate_triplet_at_position(new_s, position))
                    new_strings = temp_new_strings
                elif op_type == 'e':
                    # Convert to (coefficient, string) format for recursive call
                    temp_input = [(1, new_s) for new_s in new_strings]  # Coefficient 1 for intermediate
                    temp_result = e(temp_input, position, verbose=False)
                    # Extract just the strings (coefficients should still be 1)
                    new_strings = [string for _, string in temp_result]

            # Apply inverse bending and add to output with original coefficient
            final_results = [inverse_bending_power(new_s, i - 1) for new_s in new_strings]
            for result in final_results:
                S_out.append((coeff, result))
            if verbose:
                print(f"e({coeff} * {s}, {i}) -> {[f'{coeff} * {res}' for res in final_results]} (splitting case)")

    return S_out



def f(S, i, verbose=False):
    """
    Apply operation f_i to a list of (coefficient, string) pairs.
    f_i(S) = e_i e_{i+1} e_i(S) - e_i(S)

    Args:
        S: List of (coefficient, string) pairs
        i: Position for the f operation
        verbose: Whether to print detailed output

    Returns:
        List of (coefficient, string) pairs representing the formal sum
    """
    if verbose:
        print(f"Computing f({display_coefficient_strings(S)}, {i})")
        print(f"f_{i} = e_{i} e_{{{i+1}}} e_{i} - e_{i}")

    # Convert input to FormalSum for easier manipulation
    input_formal_sum = FormalSum()
    input_formal_sum.add_coefficient_string_pairs(S)

    if verbose:
        print(f"Input as formal sum: {input_formal_sum}")

    # Compute e_i e_{i+1} e_i(S)
    term1 = apply_sequence_to_formal_sum(input_formal_sum, [i, i+1, i])
    if verbose:
        print(f"e_{i} e_{{{i+1}}} e_{i}(S) = {term1}")

    # Compute e_i(S)
    term2 = apply_sequence_to_formal_sum(input_formal_sum, [i])
    if verbose:
        print(f"e_{i}(S) = {term2}")

    # Compute the difference: e_i e_{i+1} e_i(S) - e_i(S)
    result_formal_sum = term1 - term2
    if verbose:
        print(f"f_{i}(S) = e_{i} e_{{{i+1}}} e_{i}(S) - e_{i}(S) = {result_formal_sum}")

    # Convert back to list of (coefficient, string) pairs
    result_pairs = []
    for string_tuple, coeff in result_formal_sum.terms.items():
        result_pairs.append((coeff, list(string_tuple)))

    return result_pairs









# Test braid relation of e_i  generators

In [3]:


def test_braid_relation(test_cases, verbose=False):
    """Test the braid relation: e_i e_{i+1} e_i - e_i = e_{i+1} e_i e_{i+1} - e_{i+1}"""
    results = []

    for case in test_cases:
        formal_sum = case['formal_sum']
        i = case['position']
        name = case.get('name', f'Test case at position {i}')

        try:
            if verbose:
                print(f"\nTesting braid relation for {name}")
                print(f"Initial formal sum: {formal_sum}")
                print(f"Testing: e_{i} e_{{{i+1}}} e_{i} - e_{i} = e_{{{i+1}}} e_{i} e_{{{i+1}}} - e_{{{i+1}}}")

            # Left side: e_i e_{i+1} e_i - e_i
            left_term1 = apply_sequence_to_formal_sum(formal_sum, [i, i+1, i])  # e_i e_{i+1} e_i S
            left_term2 = apply_sequence_to_formal_sum(formal_sum, [i])          # e_i S
            left_side = left_term1 - left_term2

            # Right side: e_{i+1} e_i e_{i+1} - e_{i+1}
            right_term1 = apply_sequence_to_formal_sum(formal_sum, [i+1, i, i+1])  # e_{i+1} e_i e_{i+1} S
            right_term2 = apply_sequence_to_formal_sum(formal_sum, [i+1])          # e_{i+1} S
            right_side = right_term1 - right_term2

            # Check if they're equal
            is_valid = left_side == right_side

            results.append({
                'name': name,
                'position': i,
                'initial_sum': str(formal_sum),
                'left_side': str(left_side),
                'right_side': str(right_side),
                'is_valid': is_valid
            })

            if verbose:
                print(f"  Left side (e_{i} e_{{{i+1}}} e_{i} - e_{i}): {left_side}")
                print(f"  Right side (e_{{{i+1}}} e_{i} e_{{{i+1}}} - e_{{{i+1}}}): {right_side}")
                print(f"  Relation holds: {is_valid}")
                if not is_valid:
                    print(f"  *** BRAID RELATION VIOLATED ***")

        except Exception as ex:
            results.append({
                'name': name,
                'position': i,
                'error': str(ex),
                'is_valid': None
            })
            if verbose:
                print(f"  Error: {ex}")

    return results



def create_test_cases_12_sites():
    """Create test cases using all valid strings of length 12 (n=4)"""
    n = Polynomial.variable()
    test_cases = []

    print("Generating all valid strings with 12 sites (n=4)...")
    all_valid_strings_12 = generate_all_valid_strings(4)  # 4*3 = 12 sites
    print(f"Found {len(all_valid_strings_12)} valid strings of length 12")

    # Test with different polynomial coefficients and positions
    coefficients_to_test = [
        (Polynomial.constant(1), "1")
    ]

    positions_to_test = [1, 2, 3, 4, 5]  # Test various positions

    # Limit the number of strings to test for performance (can adjust this)
    max_strings_per_coeff = 5  # Test first 5 strings for each coefficient

    for coeff, coeff_name in coefficients_to_test:
        for pos in positions_to_test:
            strings_to_test = all_valid_strings_12[:max_strings_per_coeff]

            for idx, string in enumerate(strings_to_test):
                # Create formal sum for this string
                formal_sum = FormalSum()
                formal_sum.add_term(string, coeff)

                test_cases.append({
                    'formal_sum': formal_sum,
                    'position': pos,
                    'name': f'{coeff_name} * string_{idx+1}_12sites at pos_{pos}',
                    'string_index': idx,
                    'string_length': len(string)
                })

    return test_cases

def create_comprehensive_12_site_tests():
    """Create a more comprehensive but manageable set of 12-site tests"""
    n = Polynomial.variable()
    test_cases = []

    print("Generating comprehensive 12-site test cases...")
    all_valid_strings_12 = generate_all_valid_strings(4)
    print(f"Total valid strings with 12 sites: {len(all_valid_strings_12)}")

    # Select representative strings for testing
    num_strings_to_test = len(all_valid_strings_12)
    selected_strings = all_valid_strings_12[:num_strings_to_test]

    # Test each selected string with different coefficients and positions
    coefficients = [
        Polynomial.constant(1)
    ]

    positions = [1, 2, 3]  # Test positions 1, 2, 3

    for string_idx, string in enumerate(selected_strings):
        for coeff_idx, coeff in enumerate(coefficients):
            for pos in positions:
                formal_sum = FormalSum()
                formal_sum.add_term(string, coeff)

                test_cases.append({
                    'formal_sum': formal_sum,
                    'position': pos,
                    'name': f'String{string_idx+1}_Coeff{coeff_idx+1}_Pos{pos}',
                    'detailed_name': f'({coeff}) * {string[:6]}... at position {pos}',
                    'string_index': string_idx,
                    'coefficient_index': coeff_idx
                })

    return test_cases

def run_12_site_braid_tests(comprehensive=False, verbose=False):
    """Run braid relation tests on 12-site strings"""
    print("BRAID RELATION TEST ON 12-SITE STRINGS")
    print("Testing: e_i e_{i+1} e_i - e_i = e_{i+1} e_i e_{i+1} - e_{i+1}")
    print("=" * 70)

    # Choose test set based on comprehensive flag
    if comprehensive:
        test_cases = create_comprehensive_12_site_tests()
        print("Running COMPREHENSIVE 12-site tests...")
    else:
        test_cases = create_test_cases_12_sites()
        print("Running STANDARD 12-site tests...")

    print(f"Total test cases: {len(test_cases)}")
    print("\n" + "=" * 70)

    # Run tests with progress indication
    results = []
    failed_cases = []
    error_cases = []

    for i, case in enumerate(test_cases):
        if not verbose and i % 10 == 0:  # Progress indicator
            print(f"Progress: {i+1}/{len(test_cases)} tests completed...")

        try:
            # Run individual test
            case_results = test_braid_relation([case], verbose=verbose)
            result = case_results[0]
            results.append(result)

            if result.get('is_valid') == False:
                failed_cases.append(result)
            elif result.get('is_valid') is None:
                error_cases.append(result)

        except Exception as e:
            error_case = {
                'name': case['name'],
                'position': case['position'],
                'error': str(e),
                'is_valid': None
            }
            results.append(error_case)
            error_cases.append(error_case)

    # Summarize results
    print(f"\n" + "=" * 70)
    print("SUMMARY OF 12-SITE BRAID RELATION RESULTS")
    print("=" * 70)

    passed = sum(1 for r in results if r.get('is_valid') == True)
    failed = len(failed_cases)
    errors = len(error_cases)

    print(f"Total tests: {len(results)}")
    print(f"Passed (relation holds): {passed}")
    print(f"Failed (relation violated): {failed}")
    print(f"Errors: {errors}")
    print(f"Success rate: {passed/len(results)*100:.1f}%")

    # Report failures
    if failed > 0:
        print(f"\n*** FAILED TESTS (showing first 5) ***")
        for i, r in enumerate(failed_cases[:5]):
            print(f"❌ {i+1}. {r['name']}")
            if verbose:
                print(f"   Left side:  {r['left_side']}")
                print(f"   Right side: {r['right_side']}")
            print()

        if len(failed_cases) > 5:
            print(f"... and {len(failed_cases) - 5} more failed cases")

    # Report errors
    if errors > 0:
        print(f"\n*** ERRORS (showing first 3) ***")
        for i, r in enumerate(error_cases[:3]):
            print(f"⚠️  {i+1}. {r['name']}: {r.get('error', 'Unknown error')}")

        if len(error_cases) > 3:
            print(f"... and {len(error_cases) - 3} more errors")

    # Overall result
    if failed == 0 and errors == 0:
        print(f"\n🎉 ALL 12-SITE BRAID RELATIONS SATISFIED!")
    elif failed == 0:
        print(f"\n✅ All computed braid relations satisfied (but {errors} errors occurred)")
    else:
        print(f"\n❌ {failed} braid relation violations found")

    return {
        'results': results,
        'passed': passed,
        'failed': failed_cases,
        'errors': error_cases,
        'total': len(results)
    }


results_summary = run_12_site_braid_tests(comprehensive=True, verbose=False)



BRAID RELATION TEST ON 12-SITE STRINGS
Testing: e_i e_{i+1} e_i - e_i = e_{i+1} e_i e_{i+1} - e_{i+1}
Generating comprehensive 12-site test cases...
Total valid strings with 12 sites: 462
Running COMPREHENSIVE 12-site tests...
Total test cases: 1386

Progress: 1/1386 tests completed...
Progress: 11/1386 tests completed...
Progress: 21/1386 tests completed...
Progress: 31/1386 tests completed...
Progress: 41/1386 tests completed...
Progress: 51/1386 tests completed...
Progress: 61/1386 tests completed...
Progress: 71/1386 tests completed...
Progress: 81/1386 tests completed...
Progress: 91/1386 tests completed...
Progress: 101/1386 tests completed...
Progress: 111/1386 tests completed...
Progress: 121/1386 tests completed...
Progress: 131/1386 tests completed...
Progress: 141/1386 tests completed...
Progress: 151/1386 tests completed...
Progress: 161/1386 tests completed...
Progress: 171/1386 tests completed...
Progress: 181/1386 tests completed...
Progress: 191/1386 tests completed...


# Test braid relations for f generators


In [ ]:
def test_cubic_f_relation(test_cases, verbose=False):
    """Test the cubic relation: f_i f_{i+1} f_i(S) = n² f_i(S)"""
    results = []

    for case in test_cases:
        S = case['input_pairs']
        i = case['position']
        name = case.get('name', f'Test case at position {i}')

        try:
            if verbose:
                print(f"\nTesting cubic f relation for {name}")
                print(f"Input: {display_coefficient_strings(S)}")
                print(f"Testing: f_{i} f_{{{i+1}}} f_{i}(S) = n² f_{i}(S)")

            # Compute left side: f_i f_{i+1} f_i(S)
            if verbose:
                print(f"\nComputing left side: f_{i} f_{{{i+1}}} f_{i}(S)")

            # Step 1: f_i(S)
            step1 = f(S, i, verbose=verbose)
            if verbose:
                print(f"Step 1 - f_{i}(S): {display_coefficient_strings(step1)}")

            # Step 2: f_{i+1}(f_i(S))
            step2 = f(step1, i+1, verbose=verbose)
            if verbose:
                print(f"Step 2 - f_{{{i+1}}}(f_{i}(S)): {display_coefficient_strings(step2)}")

            # Step 3: f_i(f_{i+1}(f_i(S)))
            left_side_pairs = f(step2, i, verbose=verbose)
            if verbose:
                print(f"Step 3 - f_{i}(f_{{{i+1}}}(f_{i}(S))): {display_coefficient_strings(left_side_pairs)}")

            # Convert to formal sum
            left_side = FormalSum()
            left_side.add_coefficient_string_pairs(left_side_pairs)

            # Compute right side: n² f_i(S)
            if verbose:
                print(f"\nComputing right side: n² f_{i}(S)")

            # First compute f_i(S)
            f_i_result = f(S, i, verbose=verbose)
            if verbose:
                print(f"f_{i}(S): {display_coefficient_strings(f_i_result)}")

            # Then multiply by n²
            n_squared = Polynomial.variable() * Polynomial.variable()  # n²
            right_side_pairs = []
            for coeff, string in f_i_result:
                new_coeff = coeff * n_squared
                right_side_pairs.append((new_coeff, string))

            if verbose:
                print(f"n² f_{i}(S): {display_coefficient_strings(right_side_pairs)}")

            # Convert to formal sum
            right_side = FormalSum()
            right_side.add_coefficient_string_pairs(right_side_pairs)

            # Check if they're equal
            is_valid = left_side == right_side

            results.append({
                'name': name,
                'position': i,
                'input': display_coefficient_strings(S),
                'left_side': str(left_side),
                'right_side': str(right_side),
                'is_valid': is_valid
            })

            if verbose:
                print(f"\nComparison:")
                print(f"Left side  (f_{i} f_{{{i+1}}} f_{i}(S)): {left_side}")
                print(f"Right side (n² f_{i}(S)): {right_side}")
                print(f"Cubic relation holds: {is_valid}")
                if not is_valid:
                    print(f"*** CUBIC RELATION VIOLATED ***")

        except Exception as ex:
            results.append({
                'name': name,
                'position': i,
                'error': str(ex),
                'is_valid': None
            })
            if verbose:
                print(f"Error: {ex}")

    return results

def create_12_site_cubic_test_cases():
    """Create test cases for the cubic f relation on 12-site strings"""
    n = Polynomial.variable()
    test_cases = []

    print("Generating 12-site test cases for cubic f relation...")
    all_valid_strings_12 = generate_all_valid_strings(4)  # 4*3 = 12 sites
    print(f"Found {len(all_valid_strings_12)} valid strings of length 12")

    # Select a representative sample for testing
    num_strings_to_test = len(all_valid_strings_12)  # Test first 5 strings
    selected_strings = all_valid_strings_12[:num_strings_to_test]

    # Test with different polynomial coefficients
    coefficients = [
        (Polynomial.constant(1), "1")
    ]

    # Test positions (need both i and i+1 to be valid)
    positions = [1, 2, 3, 4, 5, 6, 7, 8]  # For 12-site strings, positions up to ~8 should be safe

    for string_idx, string in enumerate(selected_strings):
        for coeff, coeff_name in coefficients:
            for pos in positions:
                test_cases.append({
                    'input_pairs': [(coeff, string)],
                    'position': pos,
                    'name': f'String{string_idx+1}_{coeff_name}_pos{pos}',
                    'string_index': string_idx,
                    'coefficient_name': coeff_name
                })

    return test_cases

def run_12_site_cubic_f_tests(verbose=False):
    """Run cubic f relation tests on 12-site strings"""
    print("CUBIC f RELATION TEST ON 12-SITE STRINGS")
    print("Testing: f_i f_{i+1} f_i(S) = n² f_i(S)")
    print("=" * 60)

    # Create test cases
    test_cases = create_12_site_cubic_test_cases()
    print(f"Total test cases: {len(test_cases)}")
    print("\n" + "=" * 60)

    # Run tests with progress indication
    results = []
    failed_cases = []
    error_cases = []

    for i, case in enumerate(test_cases):
        if not verbose and i % 50 == 0:  # Progress indicator
            print(f"Progress: {i+1}/{len(test_cases)} tests completed...")

        try:
            # Run individual test
            case_results = test_cubic_f_relation([case], verbose=verbose)
            result = case_results[0]
            results.append(result)

            if result.get('is_valid') == False:
                failed_cases.append(result)
            elif result.get('is_valid') is None:
                error_cases.append(result)

        except Exception as e:
            error_case = {
                'name': case['name'],
                'position': case['position'],
                'error': str(e),
                'is_valid': None
            }
            results.append(error_case)
            error_cases.append(error_case)

    # Summarize results
    print(f"\n" + "=" * 60)
    print("SUMMARY OF CUBIC f RELATION RESULTS")
    print("=" * 60)

    passed = sum(1 for r in results if r.get('is_valid') == True)
    failed = len(failed_cases)
    errors = len(error_cases)

    print(f"Total tests: {len(results)}")
    print(f"Passed (relation holds): {passed}")
    print(f"Failed (relation violated): {failed}")
    print(f"Errors: {errors}")
    print(f"Success rate: {passed/len(results)*100:.1f}%")

    # Report failures
    if failed > 0:
        print(f"\n*** FAILED TESTS (showing first 3) ***")
        for i, r in enumerate(failed_cases[:3]):
            print(f"❌ {i+1}. {r['name']}")
            if verbose:
                print(f"   Left side:  {r['left_side']}")
                print(f"   Right side: {r['right_side']}")
            print()

        if len(failed_cases) > 3:
            print(f"... and {len(failed_cases) - 3} more failed cases")

    # Report errors
    if errors > 0:
        print(f"\n*** ERRORS (showing first 2) ***")
        for i, r in enumerate(error_cases[:2]):
            print(f"⚠️  {i+1}. {r['name']}: {r.get('error', 'Unknown error')}")

        if len(error_cases) > 2:
            print(f"... and {len(error_cases) - 2} more errors")

    # Overall result
    if failed == 0 and errors == 0:
        print(f"\n🎉 ALL CUBIC f RELATIONS SATISFIED!")
    elif failed == 0:
        print(f"\n✅ All computed cubic relations satisfied (but {errors} errors occurred)")
    else:
        print(f"\n❌ {failed} cubic relation violations found")

    return {
        'results': results,
        'passed': passed,
        'failed': failed_cases,
        'errors': error_cases,
        'total': len(results)
    }



def analyze_cubic_f_results(results_summary):
    """Analyze the results from cubic f relation testing"""
    print("\nANALYSIS OF CUBIC f RELATION RESULTS")
    print("=" * 45)

    total = results_summary['total']
    passed = results_summary['passed']
    failed_count = len(results_summary['failed'])
    error_count = len(results_summary['errors'])

    print(f"Success rate: {passed/total*100:.1f}%")
    print(f"Failure rate: {failed_count/total*100:.1f}%")
    print(f"Error rate: {error_count/total*100:.1f}%")

    if failed_count > 0:
        print(f"\nFailed cases analysis:")
        failed_positions = {}
        failed_coeffs = {}

        for case in results_summary['failed']:
            # Analyze by position
            pos = case['position']
            failed_positions[pos] = failed_positions.get(pos, 0) + 1

            # Try to extract coefficient info from name
            name = case['name']
            if '_1_' in name:
                coeff_type = '1'
            elif '_n_' in name:
                coeff_type = 'n'
            elif '_2n + 1_' in name:
                coeff_type = '2n + 1'
            elif '_n² - 1_' in name:
                coeff_type = 'n² - 1'
            else:
                coeff_type = 'unknown'

            failed_coeffs[coeff_type] = failed_coeffs.get(coeff_type, 0) + 1

        print("Failures by position:")
        for pos, count in sorted(failed_positions.items()):
            print(f"  Position {pos}: {count} failures")

        print("Failures by coefficient type:")
        for coeff, count in sorted(failed_coeffs.items()):
            print(f"  Coefficient {coeff}: {count} failures")

# Update main execution to include cubic f tests
if __name__ == "__main__":
    # Test cubic f relation examples


    print("\n" + "=" * 80 + "\n")

    # Run comprehensive cubic f tests on 12-site strings
    cubic_results = run_12_site_cubic_f_tests(verbose=False)

    # Analyze cubic f results
    analyze_cubic_f_results(cubic_results)

    print("\n" + "=" * 80 + "\n")

    # Also run the previous tests for comparison
    print("RUNNING PREVIOUS BRAID RELATION TESTS FOR COMPARISON:")
    results_summary = run_12_site_braid_tests(comprehensive=False, verbose=False)



CUBIC f RELATION TEST ON 12-SITE STRINGS
Testing: f_i f_{i+1} f_i(S) = n² f_i(S)
Generating 12-site test cases for cubic f relation...
Found 462 valid strings of length 12
Total test cases: 3696

Progress: 1/3696 tests completed...
Progress: 51/3696 tests completed...
Progress: 101/3696 tests completed...
Progress: 151/3696 tests completed...
Progress: 201/3696 tests completed...
Progress: 251/3696 tests completed...
Progress: 301/3696 tests completed...
Progress: 351/3696 tests completed...
Progress: 401/3696 tests completed...
Progress: 451/3696 tests completed...
Progress: 501/3696 tests completed...
Progress: 551/3696 tests completed...
Progress: 601/3696 tests completed...
Progress: 651/3696 tests completed...
Progress: 701/3696 tests completed...
Progress: 751/3696 tests completed...
Progress: 801/3696 tests completed...
Progress: 851/3696 tests completed...
Progress: 901/3696 tests completed...
Progress: 951/3696 tests completed...
Progress: 1001/3696 tests completed...
Progre